# FoundML 3 — Coding Assignment (Student Version)
## Identity Consistency with Competitive Learning (Olivetti Faces)

You will use **competitive learning** (online k-means with prototypes) and the structure of the Olivetti dataset
(**40 identities × 10 images**) to measure how consistently images of the **same person** map to the same prototype.

Difficulty level: **moderate**.
- You will write short pieces of code (loops, indexing, basic statistics).
- You will not implement k-means itself (we provide a correct online setup for `MiniBatchKMeans`).

This notebook contains **auto-gradable questions**. All numerical answers must be rounded to **2 decimals**.


## Rules
- Do **not** change the provided helper function `fit_online_kmeans`.
- Use the fixed seeds and hyperparameters stated in the questions.
- Labels (`y`) are used **only for evaluation**, never during learning.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_olivetti_faces
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score

np.set_printoptions(precision=6, suppress=True)


## 1. Load and inspect the Olivetti dataset

We work with the Olivetti dataset, that contains a set of face images taken between April 1992 and April 1994 at the AT&T Laboratories Cambridge. There are ten different images of each of 40 distinct subjects. For some subjects, the images were taken at different times, varying the lighting, facial expressions (open / closed eyes, smiling / not smiling) and facial details (glasses / no glasses). All the images were taken against a dark homogeneous background with the subjects in an upright, frontal position (with tolerance for some side movement). The image is quantized to 256 grey levels and stored as unsigned 8-bit integers; the loader will convert these to floating point values on the interval [0, 1], which are easier to work with for many algorithms. The original dataset consisted of 92 x 112 sized images, while the version available here consists of 64x64 images.

In [ ]:
data = fetch_olivetti_faces(shuffle=False)
X = data.data.astype(np.float32)   # shape (400, 4096)
y = data.target.astype(int)        # 40 identities, 10 images each

X.shape, y.shape


In [ ]:
def show_faces(X_flat, indices, title):
    n = len(indices)
    fig, axes = plt.subplots(1, n, figsize=(1.5*n, 2))
    fig.suptitle(title)
    for ax, i in zip(axes, indices):
        ax.imshow(X_flat[i].reshape(64, 64), cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

for i in range(40):
    # Show all 10 images of identity i
    show_faces(X, indices=list(range(i*10,(i+1)*10)), title="Identity " + str(i) + ": 10 images")


## 2. Online k-means helper (do not modify)

We run `MiniBatchKMeans` in an **online regime** (batch size 1), using repeated calls to `partial_fit`.


In [ ]:
def fit_online_kmeans(X, n_clusters, n_epochs=8, random_state=0):
    n_clusters = int(n_clusters)
    model = MiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=1,
        max_iter=1,
        init="random",
        random_state=int(random_state),
        n_init="auto",
    )

    # Initialization batch (required by scikit-learn)
    rng0 = np.random.RandomState(int(random_state))
    init_idx = rng0.choice(len(X), size=n_clusters, replace=False)
    model.partial_fit(X[init_idx])

    # Online updates
    for epoch in range(n_epochs):
        rng = np.random.RandomState(int(random_state) + epoch)
        for i in rng.permutation(len(X)):
            model.partial_fit(X[i:i+1])

    return model


## 3. Train a competitive model (k = 9 prototypes)

**Task:** Fit the model and compute:
- `centroids` (prototype vectors)
- `z` (prototype assignment for each image)

Use:
- `k = 9`
- `n_epochs = 8`
- `random_state = 0`


In [ ]:
# Write here your own code.
# Requirements:
# - define: k, model, centroids, z
# - centroids should have shape (k, 4096)
# - z should have shape (400,)

# YOUR CODE HERE


In [ ]:
# SANITY CHECK (do not modify)
assert 'centroids' in globals(), "centroids not defined"
assert 'z' in globals(), "z not defined"
assert centroids.shape == (k, 4096), "centroids must have shape (k, 4096)"
assert z.shape == (400,), "z must have shape (400,)"
print("Sanity check passed: model training outputs have correct shapes.")

## 4. Visualize learned prototypes (all k prototypes)

In [ ]:
def show_centroids(centroids, title):
    k = len(centroids)
    n_side = int(np.ceil(np.sqrt(k)))
    fig, axes = plt.subplots(n_side, n_side, figsize=(6, 6))
    fig.suptitle(title)
    for ax, c in zip(axes.ravel(), centroids):
        ax.imshow(c.reshape(64, 64), cmap="gray")
        ax.axis("off")
    for ax in axes.ravel()[k:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Write here your own code to display prototypes.
# YOUR CODE HERE


## 5. Identity Consistency Score (ICS)

For each identity (0..39):
1. Take the 10 images of that identity.
2. Look at their prototype assignments in `z`.
3. Find the most frequent prototype and compute the fraction of images assigned to it.
4. Average this fraction over all identities.

Formally:
\[ \mathrm{ICS} = \frac{1}{40} \sum_{i=0}^{39} \max_j \frac{\#\{\text{images of identity } i \text{ assigned to prototype } j\}}{10} \]

Use the function `identity_consistency_score(z, y)` implemented below that returns ICS as a float.


In [ ]:
def identity_consistency_score(z, y):
    scores = []
    for identity in np.unique(y):
        idx = np.where(y == identity)[0]
        _, counts = np.unique(z[idx], return_counts=True)
        scores.append(np.max(counts) / len(idx))
    return float(np.mean(scores))


In [ ]:
# Write here your own code.
# Requirements:
# - y contains identity labels (0..39), 10 images each
# - z contains prototype assignments (0..k-1)

# YOUR CODE HERE


In [ ]:
# SANITY CHECK (do not modify)
assert 'identity_consistency_score' in globals(), "identity_consistency_score not defined"
test_val = identity_consistency_score(z, y)
assert 0.0 <= test_val <= 1.0, "ICS must be between 0 and 1"
print("Sanity check passed: identity_consistency_score returns valid value.")

### **Question 1**
Compute the **Identity Consistency Score (ICS)** for the trained model (k = 9, seed 0).

- Enter a single floating-point number rounded to **2 decimals**.


In [ ]:
# ANSWER Q1
# Requirements:
# - compute ics using your identity_consistency_score
# - define: Q1_identity_consistency

# YOUR CODE HERE


In [ ]:
# SANITY CHECK (do not modify)
assert 'identity_consistency_score' in globals(), "identity_consistency_score not defined"
test_val = identity_consistency_score(z, y)
assert 0.0 <= test_val <= 1.0, "ICS must be between 0 and 1"
print("Sanity check passed: identity_consistency_score returns valid value.")

## 6. Random baseline

**Task:** Create a random assignment `z_random` of length 400, with values in `{0, ..., k-1}`,
using `np.random.RandomState(0)` for reproducibility. Compute the corresponding ICS.


In [ ]:
# Write here your own code.
# Requirements:
# - define: z_random
# - compute: ics_random

# YOUR CODE HERE


In [ ]:
# SANITY CHECK (do not modify)
assert 'z_random' in globals(), "z_random not defined"
assert len(z_random) == 400, "z_random must have length 400"
print("Sanity check passed: random baseline defined correctly.")

### **Question 2**
What is the **Identity Consistency Score** for the **random assignment** (k = 9)?

- Enter a single floating-point number rounded to **2 decimals**.


In [ ]:
# ANSWER Q2
# Requirements:
# - define: Q2_random_identity_consistency (rounded to 2 decimals)

# YOUR CODE HERE


## 7. Effect of number of prototypes: compare k = 6 vs k = 9

**Task:** Train two models (seed 0, 8 epochs):
- one with `k = 6`
- one with `k = 9`

Compute ICS for both and store them in a dictionary `ics_by_k` with keys 6 and 9.


In [ ]:
# Write here your own code.
# Requirements:
# - define: ics_by_k as a dict like {6: ..., 9: ...}

# YOUR CODE HERE


### **Question 3**
For which value of k is the Identity Consistency Score higher?

Choose one:
- `6`
- `9`


In [ ]:
# ANSWER Q3
# Requirements:
# - define: Q3_best_k_for_identity_consistency (must be integer 6 or 9)

# YOUR CODE HERE


## 8. Stability to data order (online learning effect)

**Task:** Train another model with the same k = 9 and epochs = 8, but using `random_state = 1`.
Compute the **Adjusted Rand Index (ARI)** between:
- `z_seed0` (your assignments from seed 0), and
- `z_seed1` (assignments from seed 1)

Use `adjusted_rand_score` from scikit-learn.


In [ ]:
# Write here your own code.
# Requirements:
# - define: z_seed0, z_seed1, ari

# YOUR CODE HERE


In [ ]:
# SANITY CHECK (do not modify)
assert 'ari' in globals(), "ari not defined"
assert -1.0 <= ari <= 1.0, "ARI must be between -1 and 1"
print("Sanity check passed: ARI value is in valid range.")

### **Question 4**
What is the **Adjusted Rand Index (ARI)** between clusterings from seed 0 and seed 1 (k = 9)?

- Enter a single floating-point number rounded to **2 decimals**.


In [ ]:
# ANSWER Q4
# Requirements:
# - define: Q4_ari_seed0_vs_seed1 (rounded to 2 decimals)

# YOUR CODE HERE


## Submission checklist

Your notebook must define the following variables:

- `Q1_identity_consistency`
- `Q2_random_identity_consistency`
- `Q3_best_k_for_identity_consistency`
- `Q4_ari_seed0_vs_seed1`
